# 01. Очистка и подготовка корпуса


In [4]:
from pathlib import Path
import sys
import os
import re
import json
from collections import Counter

PROJECT_ROOT = Path(r"C:\Users\anmrt\Desktop\Useful shit\Диплом\graduation_thesis")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data_corpus_tex"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "figures"
SRC_DIR = PROJECT_ROOT / "src"

for p in (OUTPUT_DIR, MODELS_DIR, FIGURES_DIR, SRC_DIR):
    p.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists:", DATA_DIR.exists())

PROJECT_ROOT: C:\Users\anmrt\Desktop\Useful shit\Диплом\graduation_thesis
DATA_DIR: C:\Users\anmrt\Desktop\Useful shit\Диплом\graduation_thesis\data_corpus_tex
DATA_DIR exists: True


In [ ]:
# python -m pip install spacy pandas gensim nltk
# python -m spacy download ru_core_news_sm

import pandas as pd
from gensim import corpora

try:
    import nltk
    nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords
    NLTK_STOPWORDS = set(stopwords.words("russian"))
except Exception as e:
    NLTK_STOPWORDS = set()
    print("NLTK stopwords не загружены")
    print("Причина:", repr(e))

try:
    import spacy
except ImportError as e:
    raise ImportError(
        "spaCy не установлен в текущем окружении."
    ) from e

try:
    NLP = spacy.load("ru_core_news_sm", disable=["parser", "ner"])
    print("spaCy модель загружена: ru_core_news_sm")
except OSError as e:
    raise OSError(
        "Модель spaCy ru_core_news_sm не найдена"
    ) from e


spaCy модель загружена: ru_core_news_sm


In [ ]:
CUSTOM_STOPWORDS = {
    "это", "такой", "такая", "такое", "такие", "который", "которая", "которое", "которые",
    "некоторый", "некоторая", "некоторое", "некоторые", "данный", "данная", "данное", "данные",
    "являться", "является", "быть", "иметь", "один", "два", "также", "например", "таким", "образом",
    "следующий", "следовательно", "получить", "полученный", "рассмотреть", "рассматриваемый",
    "показать", "доказать", "доказательство", "теорема", "лемма", "утверждение", "следствие",
    "работа", "статья", "раздел", "рисунок", "таблица", "формула", "случай", "условие",
    "задача", "решение", "результат", "значение", "вид", "тип", "номер", "страница"
}

STOPWORDS = NLTK_STOPWORDS | CUSTOM_STOPWORDS

LATEX_ENVIRONMENTS_TO_DROP = [
    "equation", "equation*", "align", "align*", "multline", "multline*", "gather", "gather*",
    "array", "matrix", "pmatrix", "bmatrix", "vmatrix", "cases", "thebibliography", "figure", "table"
]

LATEX_COMMANDS_KEEP_ARGUMENT = [
    "title", "section", "subsection", "subsubsection", "paragraph", "caption", "textbf", "textit", "emph"
]


def strip_latex(text: str) -> str:
    # Удаляет LaTeX-разметку и формулы, сохраняя полезный текст из заголовков/секций
    text = re.sub(r"%.*", " ", text)  # комментарии

    # Удаление окружения с формулами/таблицами/библиографией целиком
    for env in LATEX_ENVIRONMENTS_TO_DROP:
        pattern = rf"\\begin\{{{re.escape(env)}\}}.*?\\end\{{{re.escape(env)}\}}"
        text = re.sub(pattern, " ", text, flags=re.DOTALL | re.IGNORECASE)

    # Удаление inline/display math
    text = re.sub(r"\$\$.*?\$\$", " ", text, flags=re.DOTALL)
    text = re.sub(r"\$.*?\$", " ", text, flags=re.DOTALL)
    text = re.sub(r"\\\[.*?\\\]", " ", text, flags=re.DOTALL)
    text = re.sub(r"\\\(.*?\\\)", " ", text, flags=re.DOTALL)

    # Сохранение содержимого важных команд: \section{...} -> ...
    for cmd in LATEX_COMMANDS_KEEP_ARGUMENT:
        text = re.sub(rf"\\{cmd}\*?\{{([^{{}}]*)\}}", r" \1 ", text)

    # Удаление команды с аргументами: \command[...]{...}
    text = re.sub(r"\\[a-zA-Zа-яА-Я]+\*?(\[[^\]]*\])?(\{[^{}]*\})?", " ", text)
    # Удаление одиночных команды/экранированных символов
    text = re.sub(r"\\[^\s]", " ", text)
    text = re.sub(r"[{}_^~&#]", " ", text)
    return text


def normalize_text_for_spacy(text: str) -> str:
    # Убрать буквенно-цифровой мусор и оставить русский текст
    text = text.lower().replace("ё", "е")
    # Удаляем всё, что содержит цифры или латиницу: 0040i, 2xuku, 1127–1136, 0u0
    text = re.sub(r"\b[\wа-яА-ЯёЁ]*[0-9a-zA-Z][\wа-яА-ЯёЁ]*\b", " ", text)
    # Оставить только кириллицу и пробелы
    text = re.sub(r"[^а-яА-ЯёЁ\s-]", " ", text)
    text = re.sub(r"[-–—]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_and_tokenize(text: str) -> list[str]:
    text = normalize_text_for_spacy(text)
    if not text:
        return []

    doc = NLP(text)

    tokens = []
    for token in doc:
        # spaCy дополнительно отсекает пунктуацию/пробелы, если они вдруг остались
        if token.is_space or token.is_punct:
            continue

        raw = token.text.lower().replace("ё", "е")
        lemma = token.lemma_.lower().replace("ё", "е").strip()

        # Иногда spaCy возвращает пустую лемму или странный символ - берется исходный токен
        if not lemma or lemma == "-pron-":
            lemma = raw

        if raw in STOPWORDS or lemma in STOPWORDS:
            continue
        if len(lemma) < 3:
            continue
        if not re.fullmatch(r"[а-яе]+", lemma):
            continue

        tokens.append(lemma)

    return tokens


def preprocess_text(text: str) -> list[str]:
    return normalize_and_tokenize(strip_latex(text))


def read_tex_corpus(data_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(data_dir.rglob("*.tex")):
        raw = path.read_text(encoding="utf-8", errors="ignore")
        tokens = preprocess_text(raw)
        rows.append({
            "filename": path.name,
            "path": str(path),
            "raw_text": raw,
            "tokens": tokens,
            "clean_text": " ".join(tokens),
            "token_count": len(tokens),
        })
    return pd.DataFrame(rows)


def corpus_diagnostics(df: pd.DataFrame) -> pd.DataFrame:
    counter = Counter()
    for tokens in df["tokens"]:
        counter.update(tokens)
    return pd.DataFrame(counter.most_common(), columns=["token", "freq"])

print("Функции очистки загружены: preprocess_text, read_tex_corpus, corpus_diagnostics")
print("Лемматизация: spaCy ru_core_news_sm")

Функции очистки загружены: preprocess_text, read_tex_corpus, corpus_diagnostics
Лемматизация: spaCy ru_core_news_sm


In [7]:
df = read_tex_corpus(DATA_DIR)

print("Найдено .tex документов:", len(df))
if len(df) == 0:
    raise FileNotFoundError(f"В папке {DATA_DIR} не найдено .tex файлов. Проверь путь DATA_DIR.")

display(df[["filename", "token_count"]].head())

Найдено .tex документов: 6479


,filename,token_count
0,00-5-2001.tex,916
1,00-5.TEX,916
2,00-7-2003.tex,812
3,00-7.TEX,812
4,01-01-2000.tex,377


In [ ]:
# Диагностика длины документов после очистки
print("Документов:", len(df))
print("Пустых документов:", int((df["token_count"] == 0).sum()))
print("Средняя длина документа:", round(df["token_count"].mean(), 2))
print("Минимальная длина:", int(df["token_count"].min()))
print("Максимальная длина:", int(df["token_count"].max()))

MIN_TOKENS_PER_DOC = 20
before = len(df)
df = df[df["token_count"] >= MIN_TOKENS_PER_DOC].reset_index(drop=True)
print(f"Документов после удаления слишком коротких (<{MIN_TOKENS_PER_DOC} токенов): {len(df)} из {before}")

Документов: 6479
Пустых документов: 62
Средняя длина документа: 576.07
Минимальная длина: 0
Максимальная длина: 5787
Документов после удаления слишком коротких (<20 токенов): 6385 из 6479


In [9]:
diagnostics = corpus_diagnostics(df)
print("Топ-50 токенов после очистки:")
display(diagnostics.head(50))

bad = diagnostics[diagnostics["token"].str.contains(r"\d|[a-zA-Z]", regex=True, na=False)]
print("Количество токенов с цифрами/латиницей в частотном списке:", len(bad))
display(bad.head(30))

Топ-50 токенов после очистки:


,token,freq
0,функция,55836
1,уравнение,38389
2,система,35207
3,пространство,29253
4,точка,27431
5,множество,25644
6,число,21619
7,пусть,21112
8,оператор,19460
9,метод,19225


Количество токенов с цифрами/латиницей в частотном списке: 0


,token,freq


In [ ]:
# Словарь и BoW-корпус для LDA/LSI/BigARTM.
# no_below должен быть мягким. После хорошей очистки словарь меньше, и no_below=3/5 может удалить всё.
processed_texts = df["tokens"].tolist()

dictionary = corpora.Dictionary(processed_texts)
print("Размер словаря до filter_extremes:", len(dictionary))

# Адаптивная фильтрация: сначала используется no_below=2, если словарь обнулился - откат на no_below=1.
dictionary_candidate = corpora.Dictionary(processed_texts)
dictionary_candidate.filter_extremes(no_below=2, no_above=0.90, keep_n=10000)

if len(dictionary_candidate) == 0:
    print("Предупреждение: no_below=2 удалил весь словарь. Использую no_below=1, no_above=0.95.")
    dictionary.filter_extremes(no_below=1, no_above=0.95, keep_n=15000)
else:
    dictionary = dictionary_candidate

corpus = [dictionary.doc2bow(tokens) for tokens in processed_texts]

print("Размер словаря после filter_extremes:", len(dictionary))
print("Количество документов в BoW-корпусе:", len(corpus))
print("Пустых BoW-документов:", sum(len(doc) == 0 for doc in corpus))

Размер словаря до filter_extremes: 88678
Размер словаря после filter_extremes: 10000
Количество документов в BoW-корпусе: 6385
Пустых BoW-документов: 0


In [11]:
# Сохранение результатов
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

dictionary.save(str(MODELS_DIR / "gensim_dictionary.dict"))
corpora.MmCorpus.serialize(str(MODELS_DIR / "gensim_corpus.mm"), corpus)

df_to_save = df[["filename", "path", "clean_text", "tokens", "token_count"]].copy()
df_to_save["tokens"] = df_to_save["tokens"].apply(lambda x: " ".join(x))
df_to_save.to_csv(OUTPUT_DIR / "clean_corpus.csv", index=False, encoding="utf-8-sig")

diagnostics.to_csv(OUTPUT_DIR / "token_frequencies.csv", index=False, encoding="utf-8-sig")

print("Сохранено:")
print("-", OUTPUT_DIR / "clean_corpus.csv")
print("-", OUTPUT_DIR / "token_frequencies.csv")
print("-", MODELS_DIR / "gensim_dictionary.dict")
print("-", MODELS_DIR / "gensim_corpus.mm")

Сохранено:
- C:\Users\anmrt\Desktop\Useful shit\Диплом\graduation_thesis\outputs\clean_corpus.csv
- C:\Users\anmrt\Desktop\Useful shit\Диплом\graduation_thesis\outputs\token_frequencies.csv
- C:\Users\anmrt\Desktop\Useful shit\Диплом\graduation_thesis\models\gensim_dictionary.dict
- C:\Users\anmrt\Desktop\Useful shit\Диплом\graduation_thesis\models\gensim_corpus.mm
